In [37]:
import ast
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

from functools import reduce

In [3]:
# Convert data from byte into datatpyes
def convert_from_byte(byte_dict):
    return {key.decode('utf-8'): value.decode('utf-8') for key, value in byte_dict.items()}

In [4]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [5]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [6]:
# Only do this once
# Import VITC daraset
with open("vitc_evaluation_sup_ref.json") as f:
    vitc = json.load(f)

# Randomise order
np.random.shuffle(vitc)
# Test labels
labels = []
for claim in vitc:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([217, 283]))

In [ ]:
# Only do this once
# Populate Vercel KV with vitc datasets
for datapoint in vitc:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': datapoint['label']
    })

In [ ]:
# Only do this once
vitc_ids = [datapoint['claim_id'] for datapoint in vitc]

In [ ]:
# Only do this once
# Save ids as json 
with open('vitc_ids.json', 'w') as f:
    json.dump(vitc_ids, f)

In [84]:
# OK after randomising once at so on, use vitc ids saved in json
with open('vitc_ids.json') as f:
    vitc_ids = json.load(f)

In [85]:
# Create three batches for VITC
# 100 samples are included in all batches to test for inter-rater reliability

repeated_samples = vitc_ids[:100]

vitc_batches = {
    'vitc_repeated': repeated_samples 
}
start_index = 100
for i in range(3):
    end_index = start_index + ((len(vitc_ids) - 100) // 3) if i < 2 else len(vitc_ids)
    print(f'{start_index} - {end_index}')
    unique_samples = vitc_ids[start_index:end_index]
    start_index = end_index
    vitc_batches[f'vitc{i+1}_workpackage1'] = unique_samples[:15]
    vitc_batches[f'vitc{i+1}_workpackage3'] = unique_samples[15:]

for key in vitc_batches:
    print(key, len(vitc_batches[key])) 

100 - 233
233 - 366
366 - 500
vitc_repeated 100
vitc1_workpackage1 15
vitc1_workpackage3 118
vitc2_workpackage1 15
vitc2_workpackage3 118
vitc3_workpackage1 15
vitc3_workpackage3 119


In [16]:
# Populate vercel KV with VITC batches
for batch_id in vitc_batches.keys():
    claim_ids = vitc_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [ ]:
# Assign batches to annotators
vitc_annotators = {
    'test': 'vitc1',
    'mahmud': 'vitc1',
    'sara': 'vitc1',
    'chris': 'vitc2',
    'vishal': 'vitc3',
}

In [87]:
# Upload annotators to Vercel KV
for annotator_id in vitc_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{vitc_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'vitc_repeated',
            'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [22]:
r.hset('mahmud', mapping={'workpackage1': 'vitc1', 'stage': 'workpackage1', 'workpackage1_progress': 0, 'workpackage2_progress': 0, 'workpackage3_progress': 0})

0

In [24]:
# Replace vitc_15319 with new datapoint, but keep id the same
replacement_data = {
    "claim": "Fibromyalgia can make it hard to get out of bed.",
    "evidence": "Fibromyalgia is a medical condition defined by the presence of chronic widespread pain, fatigue, waking unrefreshed, cognitive symptoms, lower abdominal pain or cramps, and depression. Other symptoms include insomnia and a general hypersensitivity. The cause of fibromyalgia is unknown, but is believed to involve a combination of genetic and environmental factors.",
    "label": "SUPPORTS"
}

r.hset('vitc_15319', mapping=replacement_data)

0

## Phemplus

In [150]:
phemeplus_annotators = {
    'bleiz': 'phemeplus1',
    'nelly': 'phemeplus2',
    'yazhou': 'phemeplus3'
}

In [135]:


# Import phemeplus dataset
with open("phemeplus_incomplete_15-11-24.json") as f:
    phemeplus = json.load(f)

# Randomise order
np.random.shuffle(phemeplus)
# Test labels
labels = []
for claim in phemeplus:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array([False,  True]), array([ 91, 202]))

In [124]:
# Populate Vercel KV with phemeplus datasets
for datapoint in phemeplus:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': "true" if datapoint['label'] else "false"
    })

In [138]:
phemeplus_ids = [datapoint['claim_id'] for datapoint in phemeplus]

In [145]:
# Create three batches for PHEMEPLUS workpackage 1 (phemeplus is incomplete so far)
repeated_samples = phemeplus_ids[:100]
workpackage1_samples1 = phemeplus_ids[100:115]
workpackage1_samples2 = phemeplus_ids[115:130]
workpackage1_samples3 = phemeplus_ids[130:145]

phemeplus_batches = {
    'phemeplus_repeated': repeated_samples,
    'phemeplus1_workpackage1': workpackage1_samples1,
    'phemeplus2_workpackage1': workpackage1_samples2,
    'phemeplus3_workpackage1': workpackage1_samples3
}

for key in phemeplus_batches:
    print(key, len(phemeplus_batches[key]))


phemeplus_repeated 100
phemeplus1_workpackage1 15
phemeplus2_workpackage1 15
phemeplus3_workpackage1 15


In [147]:
# Populate vercel KV with phemeplus batches
for batch_id in phemeplus_batches.keys():
    claim_ids = phemeplus_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [151]:
# Upload annotators to Vercel KV
for annotator_id in phemeplus_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{phemeplus_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'phemeplus_repeated',
            # 'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

## Climate Fever

In [6]:
# Import VITC daraset
with open("clfever.json") as f:
    clfever = json.load(f)

# Randomise order
np.random.shuffle(clfever)
# Test labels
labels = []
for claim in clfever:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([143, 357]))

In [ ]:
# Only do this once
# Populate Vercel KV with vitc datasets
for datapoint in clfever:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': datapoint['label']
    })

In [7]:
# Only do this once
clfever_ids = [datapoint['claim_id'] for datapoint in clfever]

In [8]:
# Only do this once
# Save ids as json 
with open('clfever_ids.json', 'w') as f:
    json.dump(clfever_ids, f)

In [9]:
# OK after randomising once at so on, use vitc ids saved in json
with open('clfever_ids.json') as f:
    clfever_ids = json.load(f)

In [12]:
# Create three batches for Climate Fever
# 100 samples are included in all batches to test for inter-rater reliability

repeated_samples = clfever_ids[:100]

clfever_batches = {
    'clfever_repeated': repeated_samples 
}
start_index = 100
for i in range(3):
    end_index = start_index + ((len(clfever_ids) - 100) // 3) if i < 2 else len(clfever_ids)
    print(f'{start_index} - {end_index}')
    unique_samples = clfever_ids[start_index:end_index]
    start_index = end_index
    clfever_batches[f'clfever{i+1}_workpackage1'] = unique_samples[:15]
    clfever_batches[f'clfever{i+1}_workpackage3'] = unique_samples[15:]

for key in clfever_batches:
    print(key, len(clfever_batches[key])) 

100 - 233
233 - 366
366 - 500
clfever_repeated 100
clfever1_workpackage1 15
clfever1_workpackage3 118
clfever2_workpackage1 15
clfever2_workpackage3 118
clfever3_workpackage1 15
clfever3_workpackage3 119


In [13]:
# Populate vercel KV with Climate Fever batches
for batch_id in clfever_batches.keys():
    claim_ids = clfever_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [18]:
# Assign batches to annotators
clfever_annotators = {
    'clfever_test': 'clfever1',
    'yuli': 'clfever1',
    'jorge': 'clfever2',
    'anel': 'clfever3',    
}

In [19]:
# Upload annotators to Vercel KV
for annotator_id in clfever_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{clfever_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'clfever_repeated',
            'workpackage3': f'{clfever_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [21]:
r.hset('clfever_test', mapping={'stage': 'workpackage1', 'workpackage1_progress': 0, 'workpackage2_progress': 0, 'workpackage3_progress': 0})

0

## save data

In [23]:
# check submission 
test_data = r.hgetall('yazhou')
test_data = convert_from_byte(test_data)
test_data['workpackage2_progress']

'100'

In [24]:
def save_data(id, workpackage_id):
    data = r.hgetall(id)
    data = convert_from_byte(data)
    # Get ids for workpackages
    workpackage_mapping = ast.literal_eval(data['stage_to_batch'])
    workpackage = workpackage_mapping[workpackage_id]
    claim_ids = ast.literal_eval(convert_from_byte(r.hgetall(workpackage))['claim_ids'])
    dataset = []
    for claim_id in claim_ids:
        if claim_id not in data:
            print(f"Claim {claim_id} not found in {id}")
            continue
        datapoint = ast.literal_eval(data[claim_id])
        # Get claim, evidence and label as well
        claim_data = convert_from_byte(r.hgetall(claim_id))
        dataset.append({
            'claim_id': claim_id,
            'claim': claim_data['claim'],
            'evidence': claim_data['evidence'],
            'label': claim_data['label'],
            'reasoning': datapoint[0],
            'explanation': datapoint[1]
        })

    with open(f'{id}_{workpackage_id}.json', 'w') as f:
        json.dump(dataset, f)
    
    return dataset

In [26]:
dataset = save_data('yazhou', 'workpackage2')

In [27]:
df = pd.DataFrame(dataset)
df['reasoning'].value_counts()

reasoning
deductive    89
abductive    11
Name: count, dtype: int64

In [ ]:
with open('chris_workpackage2.json') as f:
    chris_workpackage2 = json.load(f)

with open('sara_workpackage2.json') as f:
    sara_workpackage2 = json.load(f)

# convert to dataframe
df_chris = pd.DataFrame(chris_workpackage2)
df_sara = pd.DataFrame(sara_workpackage2)

# compare if claim_id are the same in all three datasets



In [27]:
chris_abdudctive = np.where(df_chris['reasoning'] == 'abductive')
sara_abdudctive = np.where(df_sara['reasoning'] == 'abductive')

np.intersect1d(chris_abdudctive, sara_abdudctive)

array([ 9, 18, 34, 39, 43])

In [32]:
# load bleiz and nelly workpackage 2
with open('bleiz_workpackage2.json') as f:
    bleiz_workpackage2 = json.load(f)

with open('nelly_workpackage2.json') as f:
    nelly_workpackage2 = json.load(f)

with open('yazhou_workpackage2.json') as f:
    yazhou_workpackage2 = json.load(f)

# convert to dataframe
df_bleiz = pd.DataFrame(bleiz_workpackage2)
df_nelly = pd.DataFrame(nelly_workpackage2)
df_yazhou = pd.DataFrame(yazhou_workpackage2)

# compare if claim_id are the same in all three datasets 
df_bleiz['claim_id'].equals(df_nelly['claim_id']) and df_bleiz['claim_id'].equals(df_yazhou['claim_id'])

True

In [35]:
# check overlapping values between to arrays
bleiz_abductive = np.where(df_bleiz['reasoning'] == 'abductive')
nelly_abductive = np.where(df_nelly['reasoning'] == 'abductive')
yazhou_abductive = np.where(df_yazhou['reasoning'] == 'abductive')
reduce(np.intersect1d, (nelly_abductive, bleiz_abductive, yazhou_abductive))


array([ 6, 62])

In [158]:
# compare resoning 
np.where(df_bleiz['reasoning'] != df_nelly['reasoning'])

(array([10, 14, 15, 24, 30, 39, 52, 55, 58, 60, 66, 70, 79, 83, 87, 94]),)

In [184]:
annotation = df_nelly.iloc[79]
print(annotation['claim'])
print(annotation['evidence'])
print(annotation['label'])
print(annotation['reasoning'])
print(annotation['explanation'])

Police say shots fired at 3 #Ottawa sites - National War Memorial, Parliament Hill, and now Rideau shopping centre
A third shooting took place in Ottawa, Canada on Wednesday morning at the Rideau Centre, police said. The first two shooting situations happened at Parliament Hill and War Memorial. It appears that the shots exchanged in or near the Rideau Centre may have been between police officers and a shooting suspect. 
 One gunman shot a soldier at Canada's National War Memorial before facing off with police in the parliament building Harding said in an email to Business Insider. The capital is on lockdown as police investigate shootings at two locations: the War Memorial and Parliament Hill Ottawa is under lockdown as police investigate reports of shootings in two locations: the National War Memorial and Parliament Hill Corporal Nathan Cirillo, standing guard by Canada’s National War Memorial with a rifle, A gunman opened fire at Canada's National War Memorial on Wednesday, killing 